## Descargar los datos preprocesados para probar este ejercicio

In [ ]:
# Download Ground Truth
!mkdir Dataset
!wget  https://zenodo.org/records/12751419/files/XdataV2.npy?download=1  -O Dataset/Xdata.npy
!wget  https://zenodo.org/records/12751419/files/YdataV2.npy?download=1  -O Dataset/Ydata.npy

# Import the necessary libraries to run machine learning model efficiently

In [ ]:
# import libraries
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import sklearn
import keras

os.environ["SM_FRAMEWORK"] = "tf.keras"
import tensorflow as tf
import segmentation_models as sm
from segmentation_models import Unet

keras.backend.set_image_data_format("channels_last")

El código importa y preprocesa los datos de imagen: carga arreglos de numpy y los transpone al formato que requieren los marcos de redes neuronales. Primero carga el conjunto de imágenes (`Xdata.npy`) y el de verdad de terreno (`Ydata.npy`) con `np.load()`, que lee los archivos de arreglos de numpy guardados en la carpeta "Dataset". Luego aplica `.transpose(0, 2, 3, 1)` a ambos conjuntos para reordenar sus ejes de (muestras, canales, alto, ancho) a (muestras, alto, ancho, canales), un formato común para imágenes en aprendizaje profundo. Finalmente imprime las formas de los conjuntos transpuestos para confirmar sus dimensiones: el número de muestras, el alto y el ancho de las imágenes y el número de canales.

In [ ]:
# import and divide dataset
Xdata = np.load("Dataset/Xdata.npy").transpose(0, 2, 3, 1)
Ydata = np.load("Dataset/Ydata.npy").transpose(0, 2, 3, 1)
print(f"the shape of input image matrix is {Xdata.shape}")
print(f"the shape of input GT matrix is {Ydata.shape}")

Este código visualiza una muestra del conjunto de imágenes y su verdad de terreno. Primero fija el número de muestra `n` en 1354. Luego crea una figura con dos paneles lado a lado con `plt.subplots(1, 2, figsize=(12, 6))`. El panel izquierdo (`ax[0]`) muestra el segundo canal (índice 1) de la muestra `Xdata` en el índice `n` con `ax[0].imshow(Xdata[n, :, :, 1])`. El panel derecho (`ax[1]`) muestra el primer canal (índice 0) de la verdad de terreno `Ydata` en el mismo índice con `ax[1].imshow(Ydata[n, :, :, 0])`. Por último, las etiquetas de los ejes de ambos paneles se formatean en estilo simple y sin desplazamiento con `ax[0].ticklabel_format(useOffset=False, style="plain")` y `ax[1].ticklabel_format(useOffset=False, style="plain")`, lo que mejora la lectura.

In [ ]:
n = 1354  # sample number
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(Xdata[n, :, :, 1])
ax[1].imshow(Ydata[n, :, :, 0])

ax[0].ticklabel_format(useOffset=False, style="plain")
ax[1].ticklabel_format(useOffset=False, style="plain")

Este código divide los datos de imagen preprocesados en conjuntos de entrenamiento y de prueba. La función `train_test_split` del módulo `sklearn.model_selection` separa `Xdata` (imágenes de entrada) y `Ydata` (etiquetas de verdad de terreno) en `X_train` y `X_test` para las imágenes, y en `y_train` y `y_test` para las etiquetas. El parámetro `test_size=0.3` indica que el 30 % de los datos va al conjunto de prueba y el 70 % restante al de entrenamiento. El parámetro `random_state=42` fija la semilla del generador aleatorio para que la división sea reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    Xdata, Ydata, test_size=0.3, random_state=42
)

In [ ]:
print(f" Size of XTrain is {X_train.shape}")
print(f" Size of XTest is {X_test.shape}")
print(f" Size of YTrain is {y_train.shape}")
print(f" Size of YTest is {y_test.shape}")

### Define model and Train

Este fragmento define un modelo U-Net para segmentación de imágenes con la librería `segmentation_models`. Empieza con una línea comentada que muestra otra forma de crear la U-Net con la función `sm.Unet`, con un *backbone* `resnet18`, una sola clase de salida y activación sigmoide. Luego construye el modelo llamando directamente a la función `Unet` con estos parámetros:

- `backbone_name="resnet18"`: la arquitectura del codificador, en este caso ResNet-18, muy usada por su buena capacidad de extraer rasgos.
- `classes=1`: el modelo tiene una sola clase de salida, adecuada para segmentación binaria.
- `activation="sigmoid"`: activación sigmoide en la última capa, apropiada para clasificación binaria en la que cada píxel se clasifica de forma independiente.
- `encoder_weights=None`: el codificador no usa pesos preentrenados, de modo que el modelo se entrena desde cero.
- `input_shape=(128, 128, 13)`: la forma de las imágenes de entrada, de 128x128 píxeles con 13 canales.

Así se obtiene una U-Net ajustada a la segmentación de imágenes con las dimensiones de entrada y la arquitectura indicadas. Para más detalles, ver la [documentación de segmentation_models](https://segmentation-models.readthedocs.io/en/latest/tutorial.html).

In [ ]:
# model = sm.Unet('resnet18', classes=1, activation='sigmoid')
# https://segmentation-models.readthedocs.io/en/latest/tutorial.html
model = Unet(
    backbone_name="resnet18",
    classes=1,
    activation="sigmoid",
    encoder_weights=None,
    input_shape=(128, 128, 13),
)

El modelo se configura para el entrenamiento con el optimizador Adam, una tasa de aprendizaje de 1e-4 y la función de pérdida `binary_crossentropy`, adecuada para clasificación binaria. Además, se siguen como métricas la exactitud y el área bajo la curva (AUC). Se define una función que entrena el modelo durante 10 épocas con lotes de 32. Usa un *callback* de punto de control que guarda el mejor modelo según la pérdida de validación en cada época. El entrenamiento valida con el 20 % de los datos, y los pesos de clase se ajustan para manejar el desbalance, dando más peso a la clase 1 que a la clase 0.

In [ ]:
# Configure the model for training.
# Usamos la versión "sparse" de categorical_crossentropy
# because our target data is integers.
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(),
    ],
)


def trainmodel(model, xdata, ydata):
    NUMBER_EPOCHS = 10
    filepath = "checkpointMaping"
    BATCH_SIZE = 32

    model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath,
        monitor="val_loss",
        verbose=0,
        save_best_only=True,
        save_weights_only=False,
        mode="min",
        save_freq="epoch",
        options=None,
    )
    print(type(xdata), type(ydata))
    hist = model.fit(
        x=xdata,
        y=ydata,
        epochs=NUMBER_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,  # validación automática con el 20 % de muestras aleatorias en cada época
        verbose=1,
        callbacks=[model_checkpoint_callback],
        class_weight={0: 1, 1: 5},
    )
    return hist

Se llama a la función `trainmodel` para entrenar el modelo con los datos de entrenamiento. Las imágenes de entrada `X_train` se convierten en un arreglo de NumPy con `dtype` `float32`, un tipo de dato adecuado para el entrenamiento. Las etiquetas `y_train` también se convierten en un arreglo de NumPy `float32` y se expanden en el último eje para coincidir con la forma de entrada que espera el modelo. Esta expansión es necesaria porque el modelo espera que las etiquetas tengan una forma específica para la clasificación binaria. La función entrena el modelo con estas entradas aplicando los parámetros y *callbacks* definidos antes.

In [ ]:
trainmodel(
    model,
    np.array(X_train, dtype=np.float32),
    np.expand_dims(np.array(y_train, dtype=np.float32), axis=-1),
)

El código genera predicciones para todas las imágenes del conjunto de validación con el método `model.predict` sobre `X_test`. Este método ejecuta el modelo entrenado sobre los datos de prueba para producir las predicciones. El resultado, `val_preds`, contiene las probabilidades o etiquetas predichas para cada imagen, según la configuración del modelo. Este paso es esencial para evaluar el desempeño del modelo con datos no vistos y para calcular métricas como la exactitud o el AUC en el conjunto de validación.

In [ ]:
# Generar las predicciones para todas las imágenes del conjunto de validación

val_preds = model.predict(X_test)

Este segmento procesa las predicciones del modelo y calcula la exactitud de la clasificación.
1. **Umbralización binaria:**
   - `preds = val_preds` asigna las predicciones del modelo (`val_preds`) a la variable `preds`.
   - `preds[preds > 0.50] = 1` pone en 1 los valores predichos mayores que 0.50; así las probabilidades se convierten en etiquetas binarias con un umbral de 0.50.
   - `preds[preds <= 0.50] = 0` pone en 0 los valores menores o iguales a 0.50 y completa la clasificación binaria.

2. **Aplanado y cálculo de la exactitud:**
   - `sklearn.metrics.accuracy_score(y_test.flatten(), preds.flatten())` calcula la exactitud de las predicciones. `y_test` y `preds` se aplanan para que sean arreglos unidimensionales, como lo exige la función `accuracy_score` de `sklearn.metrics`. La función compara las etiquetas reales y las predicciones aplanadas y calcula la proporción de aciertos.

In [ ]:
preds = val_preds
preds[preds > 0.50] = 1
preds[preds <= 0.50] = 0
sklearn.metrics.accuracy_score(y_test.flatten(), preds.flatten())

Este segmento procesa las predicciones del modelo y calcula el puntaje F1 para evaluar la clasificación.

1. **Umbralización binaria:**
   - `preds = val_preds` asigna las predicciones del modelo (`val_preds`) a la variable `preds`.
   - `preds[preds > 0.50] = 1` pone en 1 los valores mayores que 0.50, convirtiendo las probabilidades en etiquetas binarias con un umbral de 0.50.
   - `preds[preds <= 0.50] = 0` pone en 0 los valores menores o iguales a 0.50 y completa la conversión a etiquetas binarias.

2. **Cálculo del puntaje F1:**
   - `sklearn.metrics.f1_score(y_test.flatten(), preds.flatten())` calcula el puntaje F1 comparando las etiquetas reales aplanadas (`y_test`) con las predicciones aplanadas (`preds`). El F1 es la media armónica de la precisión y la exhaustividad, una medida equilibrada del desempeño muy útil con conjuntos de datos desbalanceados. El aplanado deja los arreglos en el formato unidimensional que requiere la función `f1_score` de `sklearn.metrics`.

In [ ]:
preds = val_preds
preds[preds > 0.50] = 1
preds[preds <= 0.50] = 0
sklearn.metrics.f1_score(y_test.flatten(), preds.flatten())

Este código visualiza con matplotlib una muestra de las predicciones de validación y su verdad de terreno.

1. **Selección de la muestra:**
   - `n = 235` indica el índice de la muestra que se visualiza.

2. **Creación de los paneles:**
   - `fig, ax = plt.subplots(1, 2, figsize=(12, 6))` crea una figura con dos paneles lado a lado, de 12 por 6 pulgadas.

3. **Visualización de las imágenes:**
   - `ax[0].imshow(val_preds[n, :, :, :3].transpose((0, 1, 2)), vmin=0, vmax=0.5, cmap="plasma")` muestra la predicción para la muestra `n` en el primer panel (`ax[0]`), con la paleta `plasma` y una escala de color de 0 a 0.5. `.transpose((0, 1, 2))` ordena las dimensiones como (alto, ancho, canales) para mostrarla bien.
   - `ax[1].imshow(y_test[n, :, :, 0], cmap="plasma")` muestra la verdad de terreno de la misma muestra en el segundo panel (`ax[1]`), también con la paleta `plasma`.

4. **Formato de las etiquetas de los ejes:**
   - `ax[0].ticklabel_format(useOffset=False, style="plain")` y `ax[1].ticklabel_format(useOffset=False, style="plain")` muestran las etiquetas en formato simple, sin notación científica ni desplazamientos.

5. **Barras de color:**
   - `fig.colorbar(im1, ax=ax[0])` agrega una barra de color al primer panel con la escala de los valores predichos.
   - `fig.colorbar(im2, ax=ax[1])` agrega una barra de color al segundo panel con la escala de la verdad de terreno.

Esta visualización permite comparar directamente las predicciones del modelo con la verdad de terreno de una muestra, lo que ayuda a evaluar su desempeño.

In [ ]:
n = 235  # sample number
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
im1 = ax[0].imshow(
    val_preds[n, :, :, :3].transpose((0, 1, 2)), vmin=0, vmax=0.5, cmap="plasma"
)
im2 = ax[1].imshow(y_test[n, :, :, 0], cmap="plasma")

ax[0].ticklabel_format(useOffset=False, style="plain")
ax[1].ticklabel_format(useOffset=False, style="plain")

fig.colorbar(im1, ax=ax[0])
fig.colorbar(im2, ax=ax[1])